In [1]:
import sys
sys.path.append("/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/src")

from technical_analysis.backtest import run_backtest

In [2]:
import pandas as pd
import os

# test_dir = "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/"
# ln_volume = False
# test_filepaths = [os.path.join(
#     test_dir, x) for x in os.listdir(test_dir) if str(ln_volume) in x]

test_filepaths = [
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-15_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-14_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-13_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-12_ln=False.csv", 
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-11_ln=False.csv",
]

test_filepaths = [
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-15_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-14_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-13_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-12_ln=False.csv", 
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-11_ln=False.csv",
]



df_list = [pd.read_csv(fp) for fp in test_filepaths]
test_df = pd.concat(df_list, ignore_index=True)

In [3]:
len(test_df), len(test_df.drop_duplicates())

(65676, 65676)

In [4]:
test_filepaths

['/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-15_ln=False.csv',
 '/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-14_ln=False.csv',
 '/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-13_ln=False.csv',
 '/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-12_ln=False.csv',
 '/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-11_ln=False.csv']

# Buy Signal

In [5]:
import pandas as pd
import numpy as np

def look_backward_per_ticker(df: pd.DataFrame, column_to_shift: str, ticker_col: str = 'ticker', periods: int = 1) -> pd.Series:
    """
    Shifts a column backward by a set number of periods, isolated by ticker.
    Guarantees no data leakage between different stocks.
    """
    # Group by ticker, select the column, and apply the shift within that group
    return df.groupby(ticker_col)[column_to_shift].shift(periods)

def apply_mean_reversion_rule(df, z_score_threshold=2, body_ratio_threshold=0.7):
    condition_panic_selling = (df['IS_RED'] == True) & (df['BODY_RATIO'] >= body_ratio_threshold)
    condition_stat_extreme  = (df['close'] <= df['bb_lower']) & (df['z_score'].abs() > z_score_threshold)
    
    
    df['panic_alert'] = condition_panic_selling & condition_stat_extreme
    # df['was_previous_hour_panic'] = look_backward_per_ticker(df, column_to_shift='panic_alert', periods=1)
    # df['buy_signal'] = (df['was_previous_hour_panic'] == True) & (df['IS_RED'] == False)
    df["buy_signal"] = df["panic_alert"]

    return df

buy_df = apply_mean_reversion_rule(test_df, z_score_threshold=2)

# Backtest

In [6]:
"""
Possible Exit Reason:
- Take_Profit
- Stop_Loss
- Time_Limit_48h
- Data_Cutoff_End_of_file
- Stop_Loss_Simultaneous_Hit (TP and SL hit at the same time)
"""

backtest_df = run_backtest(
    buy_df, 
    profit_target=0.03, 
    stop_loss=0.03, 
    max_hours=48
)

In [12]:
from collections import Counter
len(backtest_df), len(backtest_df.drop_duplicates()), Counter(backtest_df["exit_reason"])

(160,
 160,
 Counter({'Take_Profit': 72,
          'Stop_Loss': 34,
          'Time_Limit_48h': 30,
          'Data_Cutoff_End_of_File': 24}))

In [9]:
win_rate = len(backtest_df[backtest_df['exit_reason'] == 'Take_Profit']) / len(backtest_df) * 100
lose_rate = len(backtest_df[backtest_df['exit_reason'] == 'Stop_Loss']) / len(backtest_df) * 100
time_limit_up = len(backtest_df[backtest_df['exit_reason'] == 'Time_Limit_48h']) / len(backtest_df) * 100
data_cutoff = len(backtest_df[backtest_df['exit_reason'] == 'Data_Cutoff_End_of_File']) / len(backtest_df) * 100

print(win_rate)
print(lose_rate)
print(time_limit_up)
print(data_cutoff)

45.0
21.25
18.75
15.0
